# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 protein mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

The dataset includes protein abundance, peptide, and modification analysis from human extracellular vesicles. All dataset entities are referenced by their `@id`.

In [ ]:
# Ensure mlcroissant library is installed (run this cell if mlcroissant isn't already installed)
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print dataset-level information
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"DOI: {dataset.metadata.identifier}")

## 2. Data Overview
### Explore Record Sets
View all available record sets (tables) in the dataset along with their fields and `@id` references for precise downstream access. For detailed entities, always refer by their `@id`.


In [ ]:
print("Available record sets and their fields:\n")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"Record Set: {rs.name} (@id: {rs.id})")
    print("  Fields/Columns:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print("\n")

## 3. Data Extraction
Load data from a specific record set by its `@id` into a Pandas DataFrame.

> **Tip:** Use the record set and field `@id` values shown above for unambiguous column access. See further for an example.


In [ ]:
# For this dataset, usually a single main table, so extract all record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record set IDs:", record_set_ids)

# Example: get the main proteins record set by its @id
# (Customize if needed; for this dataset likely only one record set)
main_record_set_id = record_set_ids[0]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[rs_id])} records from record set '{rs_id}'.")

print("\nColumns in main DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())

# Show first 5 records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing, filtering, and descriptive analysis steps, all using field and record set `@id` references.


In [ ]:
df = dataframes[main_record_set_id]

# List columns for selection, referencing by Croissant @id
print("Available columns (@id):")
for i, col in enumerate(df.columns):
    print(f"[{i}] {col}")

# Example: select a numeric field (e.g., coverage percentage or molecular weight) by its @id
# Replace these IDs with the correct ones from the record set (see earlier printout)
# For illustration, suppose the field for molecular weight is 'cr:MW', 'cr:peptide_count', etc.
numeric_field_id = None
for col in df.columns:
    norm_col = col.lower()
    if 'mw' in norm_col or 'molecular' in norm_col:  # Example match
        numeric_field_id = col
        break
if numeric_field_id is None:
    numeric_field_id = df.columns[1]  # Fall back to 2nd column if MW not found
print(f"Selected numeric field for analysis: {numeric_field_id}")

# Convert to numeric if needed
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors="coerce")

# Remove outliers and filter (example: MW > 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head(3))

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))

# Group by a categorical field (@id, e.g. protein description or type)
# Try to find a likely categorical field with a small number of unique values
group_field_id = None
for col in df.columns:
    if df[col].dtype == object and df[col].nunique() < df.shape[0] // 5:
        group_field_id = col
        break
if group_field_id is not None:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("\nNo suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field (e.g., histogram for molecular weight) and the relationship with a categorical field (e.g., boxplot by group_field).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=40, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If a group_field is available, show boxplots
if group_field_id is not None:
    top_groups = filtered_df[group_field_id].value_counts().index[:6]
    plt.figure(figsize=(12,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df[filtered_df[group_field_id].isin(top_groups)], showfliers=False)
    plt.title(f"{numeric_field_id} by {group_field_id} (top 6 groups)")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion

- The dataset was loaded and explored using Croissant's schema and `mlcroissant`.
- You can always reference dataset entities for analysis using their unique `@id` for reproducibility.
- We inspected the distribution and group statistics of a key numeric field, and visualized its variation.
- Use the Croissant metadata information and this notebook's code as a template for deeper biological or statistical analyses.
